1. Imports

In [1]:

import pandas as pd
import numpy as np
from scipy import stats
import matplotlib.pyplot as plt
import seaborn as sns
sns.set(style="whitegrid")


2. Load and preview

In [2]:


df = pd.read_csv("../data/togo-dapaong_qc.csv")
df.info()
df.head()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 525600 entries, 0 to 525599
Data columns (total 19 columns):
 #   Column         Non-Null Count   Dtype  
---  ------         --------------   -----  
 0   Timestamp      525600 non-null  object 
 1   GHI            525600 non-null  float64
 2   DNI            525600 non-null  float64
 3   DHI            525600 non-null  float64
 4   ModA           525600 non-null  float64
 5   ModB           525600 non-null  float64
 6   Tamb           525600 non-null  float64
 7   RH             525600 non-null  float64
 8   WS             525600 non-null  float64
 9   WSgust         525600 non-null  float64
 10  WSstdev        525600 non-null  float64
 11  WD             525600 non-null  float64
 12  WDstdev        525600 non-null  float64
 13  BP             525600 non-null  int64  
 14  Cleaning       525600 non-null  int64  
 15  Precipitation  525600 non-null  float64
 16  TModA          525600 non-null  float64
 17  TModB          525600 non-nul

,Timestamp,GHI,DNI,DHI,ModA,ModB,Tamb,RH,WS,WSgust,WSstdev,WD,WDstdev,BP,Cleaning,Precipitation,TModA,TModB,Comments
0,2021-10-25 00:01,-1.3,0.0,0.0,0.0,0.0,24.8,94.5,0.9,1.1,0.4,227.6,1.1,977,0,0.0,24.7,24.4,NaN
1,2021-10-25 00:02,-1.3,0.0,0.0,0.0,0.0,24.8,94.4,1.1,1.6,0.4,229.3,0.7,977,0,0.0,24.7,24.4,NaN
2,2021-10-25 00:03,-1.3,0.0,0.0,0.0,0.0,24.8,94.4,1.2,1.4,0.3,228.5,2.9,977,0,0.0,24.7,24.4,NaN
3,2021-10-25 00:04,-1.2,0.0,0.0,0.0,0.0,24.8,94.3,1.2,1.6,0.3,229.1,4.6,977,0,0.0,24.7,24.4,NaN
4,2021-10-25 00:05,-1.2,0.0,0.0,0.0,0.0,24.8,94.0,1.3,1.6,0.4,227.5,1.6,977,0,0.0,24.7,24.4,NaN


3. Timestamp -> datetime

In [3]:


df['Timestamp'] = pd.to_datetime(df['Timestamp'], format='%Y-%m-%d %H:%M', errors='coerce')

df = df.sort_values('Timestamp').reset_index(drop=True)


 4. Replace impossible negatives for irradiance/module readings



In [4]:

for c in ['GHI','DNI','DHI','ModA','ModB']:
    df[c] = df[c].apply(lambda x: np.nan if (isinstance(x,(int,float)) and x < 0) else x)


 5. Numeric columns list


In [5]:

numeric_cols = ['GHI','DNI','DHI','ModA','ModB','Tamb','RH','WS','WSgust','WSstdev','BP','Precipitation','TModA','TModB']


 6. Summary


In [6]:

display(df[numeric_cols].describe())
display(df.isna().sum())


,GHI,DNI,DHI,ModA,ModB,Tamb,RH,WS,WSgust,WSstdev,BP,Precipitation,TModA,TModB
count,268215.000000,525600.000000,525600.000000,525600.000000,525600.000000,525600.000000,525600.000000,525600.000000,525600.000000,525600.000000,525600.000000,525600.000000,525600.000000,525600.000000
mean,454.081218,151.258469,116.444352,226.144375,219.568588,27.751788,55.013160,2.368093,3.229490,0.557740,975.915242,0.001382,32.444403,33.543330
std,319.096010,250.956962,156.520714,317.346938,307.932510,4.758023,28.778732,1.462668,1.882565,0.268923,2.153977,0.026350,10.998334,12.769277
min,0.000000,0.000000,0.000000,0.000000,0.000000,14.900000,3.300000,0.000000,0.000000,0.000000,968.000000,0.000000,13.100000,13.100000
25%,156.500000,0.000000,0.000000,0.000000,0.000000,24.200000,26.500000,1.400000,1.900000,0.400000,975.000000,0.000000,23.900000,23.600000
50%,430.300000,0.000000,2.500000,4.400000,4.300000,27.200000,59.300000,2.200000,2.900000,0.500000,976.000000,0.000000,28.400000,28.400000
75%,743.900000,246.400000,215.700000,422.525000,411.000000,31.100000,80.800000,3.200000,4.400000,0.700000,977.000000,0.000000,40.600000,43.000000
max,1424.000000,1004.500000,805.700000,1380.000000,1367.000000,41.400000,99.800000,16.100000,23.100000,4.700000,983.000000,2.300000,70.400000,94.600000


Timestamp             0
GHI              257385
DNI                   0
DHI                   0
ModA                  0
ModB                  0
Tamb                  0
RH                    0
WS                    0
WSgust                0
WSstdev               0
WD                    0
WDstdev               0
BP                    0
Cleaning              0
Precipitation         0
TModA                 0
TModB                 0
Comments         525600
dtype: int64

 7. Median imputation


In [7]:

df[numeric_cols] = df[numeric_cols].fillna(df[numeric_cols].median())


 8. Z-score outlier detection


In [8]:

z = np.abs(stats.zscore(df[numeric_cols], nan_policy='omit'))
outlier_mask = (z > 3).any(axis=1)
print("Outliers rows:", outlier_mask.sum())

df_clean = df.loc[~outlier_mask].copy()


Outliers rows: 17401


 9. Plots (examples)


In [ ]:

plt.figure(figsize=(12,5))
plt.plot(df_clean['Timestamp'], df_clean['GHI'], label='GHI')
plt.plot(df_clean['Timestamp'], df_clean['DNI'], label='DNI')
plt.plot(df_clean['Timestamp'], df_clean['DHI'], label='DHI')
plt.legend(); plt.title("TOGO - Irradiance Time Series"); plt.show()


C:\Users\Temesgen Gonfa\Projects\Assessment\solar-challenge-week0\Lib\site-packages\IPython\core\pylabtools.py:170: UserWarning: Creating legend with loc="best" can be slow with large amounts of data.
  fig.canvas.print_figure(bytes_io, **kw)


 10. Cleaning impact


In [ ]:

display(df_clean.groupby('Cleaning')[['ModA','ModB']].mean())


,ModA,ModB
Cleaning,,
0,178.563029,170.976368
1,272.539053,276.273767


 11. Save cleaned file (do not commit data CSV to git — ../data/ is ignored)


In [ ]:

df_clean.to_csv("../data/togo_clean.csv", index=False)